In [1]:
import sys
import subprocess
import site

print(f"Aktif Python Çekirdeği: {sys.executable}")
print("Yönetici (Admin) izni bariyerini aşmak için kütüphaneler User dizinine kuruluyor...")

try:
    # --user parametresi ile kurulumu C:\ProgramData yerine doğrudan Kullanıcı (User) dizinine yapıyoruz
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--user', 'biopython', 'pandas', 'tqdm'])
    print("Kurulumlar bitti. Yollar (path) güncelleniyor ve test ediliyor...")
    
    # Python'un yeni kurulan paketleri aynı oturumda görebilmesi için yolları tazeliyoruz
    import importlib
    importlib.invalidate_caches()
    
    import pandas as pd
    from Bio import Entrez
    from tqdm.notebook import tqdm
    import os
    
    os.makedirs("data", exist_ok=True)
    print("\n✅ TEST BAŞARILI: Tüm modüller sorunsuz yüklendi ve içe aktarıldı!")
    
except Exception as e:
    print(f"\n❌ HATA OLUŞTU: {e}")

Aktif Python Çekirdeği: c:\ProgramData\anaconda3\python.exe
Yönetici (Admin) izni bariyerini aşmak için kütüphaneler User dizinine kuruluyor...
Kurulumlar bitti. Yollar (path) güncelleniyor ve test ediliyor...

✅ TEST BAŞARILI: Tüm modüller sorunsuz yüklendi ve içe aktarıldı!


In [2]:
from Bio import Entrez

# PubMed API'si için e-posta adresi (Bot korumasına takılmamak için zorunlu)
Entrez.email = "255112016@kocaeli.edu.tr"

# Türkiye adresli veya Türkçe dilindeki tıbbi yayınları arayan genişletilmiş MeSH sorgusu
SEARCH_QUERY = "(Turkey[Affiliation] OR Turkish[Language]) AND (medicine[MeSH Terms] OR health[MeSH Terms])"
BATCH_SIZE = 1000
OUTPUT_FILE = "data/01_pubmed_abstracts.csv"

print("PubMed sunucularına bağlanılıyor, ön sorgu atılıyor...")

try:
    # Toplam makale sayısını bulmak için sadece 1 sonuç getiren bir ön sorgu atıyoruz
    initial_search = Entrez.esearch(db="pubmed", term=SEARCH_QUERY, retmax=1)
    record = Entrez.read(initial_search)
    total_count = int(record["Count"])
    print(f"✅ BAĞLANTI BAŞARILI! Hedefe uygun toplam makale sayısı: {total_count}")
except Exception as e:
    print(f"❌ BAĞLANTI HATASI: {e}")

PubMed sunucularına bağlanılıyor, ön sorgu atılıyor...
✅ BAĞLANTI BAŞARILI! Hedefe uygun toplam makale sayısı: 9904


In [3]:
import time
import os
import pandas as pd
from Bio import Entrez
from urllib.error import HTTPError
from tqdm.notebook import tqdm

print("Veri indirme motoru başlatılıyor. Lütfen ilerleme çubuğunu takip edin...")

# Eğer dosya daha önce oluşturulmadıysa, boş bir CSV aç ve başlıkları yaz
if not os.path.isfile(OUTPUT_FILE):
    pd.DataFrame(columns=["source", "title", "abstract", "language"]).to_csv(OUTPUT_FILE, index=False, encoding="utf-8")

# İlerleme çubuğunu (Progress Bar) başlat
with tqdm(total=total_count, desc="PubMed Verileri İndiriliyor") as pbar:
    for start in range(0, total_count, BATCH_SIZE):
        try:
            # 1. Belirtilen aralıktaki makale ID'lerini al
            search_handle = Entrez.esearch(db="pubmed", term=SEARCH_QUERY, retstart=start, retmax=BATCH_SIZE)
            search_results = Entrez.read(search_handle)
            search_handle.close()
            id_list = search_results["IdList"]
            
            if not id_list:
                break

            # 2. ID listesini kullanarak makale detaylarını (XML) çek
            fetch_handle = Entrez.efetch(db="pubmed", id=id_list, retmode="xml")
            articles = Entrez.read(fetch_handle)
            fetch_handle.close()
            
            batch_data = []
            
            # 3. XML içinden Başlık ve Abstract (Özet) kısımlarını ayıkla
            for article in articles.get('PubmedArticle', []):
                try:
                    title = article['MedlineCitation']['Article']['ArticleTitle']
                    
                    # Abstract alanının içindeki metni al (Bazen birden fazla parça olabilir, birleştiriyoruz)
                    abstract_list = article['MedlineCitation']['Article']['Abstract']['AbstractText']
                    abstract_text = " ".join([str(abs_part) for abs_part in abstract_list])
                    
                    batch_data.append({
                        "source": "PubMed",
                        "title": title,
                        "abstract": abstract_text,
                        "language": "tr/en" # PubMed genellikle İngilizce tutar, sonradan dil tespiti ile ayrıştırılabilir
                    })
                except KeyError:
                    # Abstract'ı veya başlığı eksik olan yayınları atla
                    continue
            
            # 4. Çekilen bu partiyi diske kaydet (veri kaybını önlemek için append modunda)
            if batch_data:
                df_batch = pd.DataFrame(batch_data)
                df_batch.to_csv(OUTPUT_FILE, mode='a', header=False, index=False, encoding="utf-8")
            
            # İlerleme çubuğunu güncelle
            pbar.update(len(id_list))
            
            # Sunucuyu yormamak ve ban yememek için zorunlu dinlenme süresi
            time.sleep(1.5)
            
        except HTTPError as err:
            print(f"\nSunucu hatası (HTTP {err.code}). 10 saniye beklenip devam edilecek...")
            time.sleep(10)
            continue
        except Exception as e:
            print(f"\nBeklenmeyen hata: {e}")
            break

print(f"\nİşlem Tamamlandı! Veriler başarıyla '{OUTPUT_FILE}' konumuna kaydedildi.")

Veri indirme motoru başlatılıyor. Lütfen ilerleme çubuğunu takip edin...


PubMed Verileri İndiriliyor:   0%|          | 0/9904 [00:00<?, ?it/s]


Beklenmeyen hata: IncompleteRead(781 bytes read)

İşlem Tamamlandı! Veriler başarıyla 'data/01_pubmed_abstracts.csv' konumuna kaydedildi.


In [4]:
import time
import pandas as pd
from Bio import Entrez
from urllib.error import HTTPError
import http.client
from tqdm.notebook import tqdm

REMAINING_START = 9000
print(f"Bağlantı koptuğu yerden ({REMAINING_START}. kayıttan) devam ediliyor...")

# İlerleme çubuğunu sadece kalan miktar için başlatıyoruz
with tqdm(total=(total_count - REMAINING_START), desc="Kalan Veriler İndiriliyor") as pbar:
    for start in range(REMAINING_START, total_count, BATCH_SIZE):
        try:
            # 1. Sadece kalan aralıktaki ID'leri al
            search_handle = Entrez.esearch(db="pubmed", term=SEARCH_QUERY, retstart=start, retmax=BATCH_SIZE)
            search_results = Entrez.read(search_handle)
            search_handle.close()
            id_list = search_results["IdList"]
            
            if not id_list:
                break

            # 2. XML verisini çek
            fetch_handle = Entrez.efetch(db="pubmed", id=id_list, retmode="xml")
            articles = Entrez.read(fetch_handle)
            fetch_handle.close()
            
            batch_data = []
            
            # 3. Başlık ve Özet ayıklama
            for article in articles.get('PubmedArticle', []):
                try:
                    title = article['MedlineCitation']['Article']['ArticleTitle']
                    
                    abstract_list = article['MedlineCitation']['Article']['Abstract']['AbstractText']
                    abstract_text = " ".join([str(abs_part) for abs_part in abstract_list])
                    
                    batch_data.append({
                        "source": "PubMed",
                        "title": title,
                        "abstract": abstract_text,
                        "language": "tr/en"
                    })
                except KeyError:
                    continue
            
            # 4. Ana dosyanın sonuna (append) ekle
            if batch_data:
                df_batch = pd.DataFrame(batch_data)
                df_batch.to_csv(OUTPUT_FILE, mode='a', header=False, index=False, encoding="utf-8")
            
            pbar.update(len(id_list))
            time.sleep(2) # Kalan küçük kısım için dinlenme süresini garantiye aldık
            
        except (HTTPError, http.client.IncompleteRead) as err:
            print(f"\nSunucu dalgalanması. 5 saniye beklenip tekrar denenecek...")
            time.sleep(5)
            continue
        except Exception as e:
            print(f"\nBeklenmeyen hata: {e}")
            break

print(f"\nEksik veriler tamamlandı! Dosyan şu an tam: '{OUTPUT_FILE}'")

Bağlantı koptuğu yerden (9000. kayıttan) devam ediliyor...


Kalan Veriler İndiriliyor:   0%|          | 0/904 [00:00<?, ?it/s]


Eksik veriler tamamlandı! Dosyan şu an tam: 'data/01_pubmed_abstracts.csv'
